In [4]:
import spacy
import json
from tqdm import tqdm
import os
import pymupdf  # PyMuPDF


In [5]:

def enrich_parallel_sentences(parallel_sents):
    try:
        nlp = spacy.load("en_core_web_sm")
    except Exception:
        return parallel_sents  # skip if model missing

    for sent in tqdm(parallel_sents):
        doc = nlp(sent["translation"])
        res, tense, number, genders = [], None, None, []
        plural_details, genders_details = None, []

        for tok in doc:
            morph = tok.morph.to_dict()
            res.append(
                {
                    "word": tok.text,
                    "lemma": tok.lemma_,
                    "upos": tok.pos_,
                    "tag": tok.tag_,
                    "morph": morph,
                }
            )
            if not number and morph.get("Number"):
                number = morph["Number"]
            elif morph.get("Number") == "Plur":
                number = "Plur"
                plural_details = {"word": tok.text, "morph": morph}
            if not tense and morph.get("Tense"):
                tense = morph["Tense"]
            if morph.get("Gender"):
                genders.append(morph["Gender"])
                genders_details.append({"word": tok.text, "morph": morph})

        sent["spacy_info"] = {
            "res": res,
            "tense": tense,
            "number": number,
            "plural_details": plural_details,
            "genders": genders,
            "genders_details": genders_details,
        }
        sent["source"] = sent.pop("source")
        sent["translation"] = sent.pop("translation")

    return parallel_sents


In [18]:


language = "mandan"

train_or_test = "test"

with open(f"../data/parallel_sentences/{language}_{train_or_test}_set.json", "r", encoding="utf-8") as f:
    parallel_sentences = json.load(f)

print(len(parallel_sentences))
print(parallel_sentences[0])

500
{'source': 'g.  réeho’sh', 'gloss': 'go.there=ind.m', 'translation': '‘he went there’ (Hollow 1970: 175)'}


In [19]:
# Parallel sentence cleaner
import re
from copy import deepcopy

def clean_parallel_sentences(parallel_sentences):
    parallel_sentences_cleaned = deepcopy(parallel_sentences)
    for i, sent in enumerate(parallel_sentences_cleaned):
        sent["source"] = sent["source"].strip().replace("\n", " ")
        sent["gloss"] = sent["gloss"].strip().replace("\n", " ")
        sent["translation"] = sent["translation"].strip().replace("\n", " ")

        # Remove (n) from source (where n is a number or a letter) only at the start
        sent["source"] = re.sub(r'^\s*\([a-zA-Z0-9]+\)\s*', ' ', sent["source"]).strip()

        # Remove any single letter with . after it in source (e.g., "A. ", "b. ", "A: ", etc.)
        sent["source"] = re.sub(r'\b[a-zA-Z][\.:]\s*', ' ', sent["source"]).strip()

        # Get the text within the largest pair of same quotes in translation
        # Make sure to get the largest match if nested quotes exist
        quote_patterns = [("'", "'"), ('“', '”'), ('‘', '’'), ('"', '"')]
        max_len = 0
        matched = False
        current_match = ""
        for open_q, close_q in quote_patterns:
            first_index = sent["translation"].find(open_q)
            last_index = sent["translation"].rfind(close_q)
            if first_index != -1 and last_index != -1 and last_index > first_index:
                current_len = last_index - first_index
                if current_len > max_len:
                    max_len = current_len
                    current_match = sent["translation"][first_index + 1:last_index].strip()
                    matched = True
                    print(f"Matched quotes: {open_q}{close_q} in sentence {i} -> {current_match}")
        if matched:
            sent["translation"] = current_match
        if not matched:
            sent["translation"] = sent["translation"].strip()
            # Remove leading and trailing quotes if no matched pairs found
            sent["translation"] = re.sub(r'^[“"‘\']+', '', sent["translation"])
            sent["translation"] = re.sub(r'[”"’\']+$', '', sent["translation"]).strip()
            

        # Remove extra spaces
        sent["source"] = re.sub(r'\s+', ' ', sent["source"])
        sent["gloss"] = re.sub(r'\s+', ' ', sent["gloss"])
        sent["translation"] = re.sub(r'\s+', ' ', sent["translation"])

        # Remove citation patterns in translation like [1], (see Smith 2020), etc. ONLY AT THE END
        sent["translation"] = re.sub(r'\s*\[[^\]]*\]\s*$', ' ', sent["translation"]).strip()
        sent["translation"] = re.sub(r'\s*\(see [^\)]*\)\s*$', ' ', sent["translation"]).strip()
        sent["translation"] = re.sub(r'\s*\([^\)]*et al\)\s*$', ' ', sent["translation"]).strip()
        sent["translation"] = re.sub(r'\s*\([^\)]*202[0-9][^\)]*\)\s*$', ' ', sent["translation"]).strip()
        # Remove extra spaces again after citation removal
        sent["translation"] = re.sub(r'\s+', ' ', sent["translation"]).strip()

        

        # Further cleaning can be added here
    return parallel_sentences_cleaned

In [20]:
cleaned_parallel_sentences = clean_parallel_sentences(parallel_sentences)


Matched quotes: ‘’ in sentence 0 -> he went there
Matched quotes: ‘’ in sentence 1 -> they argued about it
Matched quotes: ‘’ in sentence 2 -> let’s go home and then you will eat
Matched quotes: ‘’ in sentence 3 -> the young men saw a few buffalo
Matched quotes: ‘’ in sentence 5 -> from another village
Matched quotes: ‘’ in sentence 6 -> right here
Matched quotes: ‘’ in sentence 7 -> he is going away
Matched quotes: ‘’ in sentence 8 -> he took her ribs on both sides...
Matched quotes: ‘’ in sentence 9 -> And, there were diamond willows in a bunch there as he woke up, that Royal Chief.
Matched quotes: ‘’ in sentence 10 -> where around here are my four arrows?
Matched quotes: ‘’ in sentence 11 -> Native American(s)
Matched quotes: ‘’ in sentence 12 -> it is you
Matched quotes: ‘’ in sentence 13 -> broom’ (lit. ‘what one sweeps it with
Matched quotes: ‘’ in sentence 14 -> That old one killed something and he made me skin half of it...
Matched quotes: ‘’ in sentence 15 -> his/her clothes, 

In [21]:
for row,row_2 in zip(cleaned_parallel_sentences, parallel_sentences):
    #print(row["source"],"--", row_2["source"])
    #print(row["gloss"])
    len1 = len(row["translation"])
    len2 = len(row_2["translation"])
    print(len2-len1)
    print(row["translation"],"\n",row_2["translation"], "\n")

21
he went there 
 ‘he went there’ (Hollow 1970: 175) 

21
they argued about it 
 ‘they argued about it’ (Hollow 1973a: 24) 

21
let’s go home and then you will eat 
 ‘let’s go home and then you will eat’ (Hollow 1973a: 87) 

21
the young men saw a few buffalo 
 ‘the young men saw a few buffalo’ (Hollow 1973b: 79) 

1
His staff had fallen down when he was laying there, [for] he, First 
 ‘His staff had fallen down when he was laying there, [for] he, First 

22
from another village 
 ‘from another village’ (Hollow 1973b: 138) 

22
right here 
 ‘right here’ (Hollow 1973a: 183) 

21
he is going away 
 ‘he is going away’ (Hollow 1970: 265) 

22
he took her ribs on both sides... 
 ‘he took her ribs on both sides...’ (Hollow 1973a: 176) 

2
And, there were diamond willows in a bunch there as he woke up, that Royal Chief. 
 ‘And, there were diamond willows in a bunch there as he woke up, that Royal Chief.’ 

21
where around here are my four arrows? 
 ‘where around here are my four arrows?’ (Ho

In [22]:
# Store cleaned data
output_file = f"../data/parallel_sentences/{language}_{train_or_test}_set_cleaned.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(cleaned_parallel_sentences, f, ensure_ascii=False, indent=4)

print(f"Saved cleaned data to {output_file}")

Saved cleaned data to ../data/parallel_sentences/mandan_test_set_cleaned.json


enriched_parallel_sentences = enrich_parallel_sentences(cleaned_parallel_sentences)

In [16]:
print(enriched_parallel_sentences[0])

{'gloss': 'pv.ins-ins.frce-be.twisted', 'spacy_info': {'res': [{'word': 'to', 'lemma': 'to', 'upos': 'PART', 'tag': 'TO', 'morph': {}}, {'word': 'twist', 'lemma': 'twist', 'upos': 'VERB', 'tag': 'VB', 'morph': {'VerbForm': 'Inf'}}, {'word': 'something', 'lemma': 'something', 'upos': 'PRON', 'tag': 'NN', 'morph': {'Number': 'Sing', 'PronType': 'Ind'}}, {'word': 'up', 'lemma': 'up', 'upos': 'ADP', 'tag': 'IN', 'morph': {}}], 'tense': None, 'number': 'Sing', 'plural_details': None, 'genders': [], 'genders_details': []}, 'source': '[ˈikãm ĩ nĩ]', 'translation': 'to twist something up'}


In [17]:
# Store in data
with open(f"../data/parallel_sentences/{language}_parallel_sentences_cleaned_enriched.json", "w", encoding="utf-8") as f:
    json.dump(enriched_parallel_sentences, f, ensure_ascii=False, indent=4)

print(f"Stored enriched parallel sentences for {language} in file: data/parallel_sentences/{language}_parallel_sentences_cleaned_enriched.json")

Stored enriched parallel sentences for mandan in file: data/parallel_sentences/mandan_parallel_sentences_cleaned_enriched.json
